# 07 EPA Features

This notebook adds Expected Points Added, also called EPA, features to the NFL forecasting model.

The previous best model used basic scoring, win percentage, and recent-form features. This notebook adds play-by-play efficiency features so the model can evaluate how efficient each team has been before each game.

Previous best accuracy: 62.46%

In [1]:
# Import packages
import os
import pandas as pd
import nfl_data_py as nfl

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
# Load play-by-playh data
seasons = list(range(2018, 2026))

pbp = nfl.import_pbp_data(seasons)

pbp.head()

2018 done.
2019 done.
2020 done.
2021 done.
2022 done.
2023 done.
2024 done.
2025 done.
Downcasting floats.


,play_id,game_id,old_game_id_x,home_team,away_team,season_type,week,posteam,posteam_type,defteam,...,was_pressure,route,defense_man_zone_type,defense_coverage_type,offense_names,defense_names,offense_positions,defense_positions,offense_numbers,defense_numbers
0,1.0,2018_01_ATL_PHI,2018090600,PHI,ATL,REG,1,None,None,None,...,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN
1,37.0,2018_01_ATL_PHI,2018090600,PHI,ATL,REG,1,ATL,away,PHI,...,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN
2,52.0,2018_01_ATL_PHI,2018090600,PHI,ATL,REG,1,ATL,away,PHI,...,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN
3,75.0,2018_01_ATL_PHI,2018090600,PHI,ATL,REG,1,ATL,away,PHI,...,False,HITCH,ZONE_COVERAGE,COVER_3,NaN,NaN,NaN,NaN,NaN,NaN
4,104.0,2018_01_ATL_PHI,2018090600,PHI,ATL,REG,1,ATL,away,PHI,...,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Check play-by-play columns
pbp.shape
pbp.columns.tolist()

['play_id',
 'game_id',
 'old_game_id_x',
 'home_team',
 'away_team',
 'season_type',
 'week',
 'posteam',
 'posteam_type',
 'defteam',
 'side_of_field',
 'yardline_100',
 'game_date',
 'quarter_seconds_remaining',
 'half_seconds_remaining',
 'game_seconds_remaining',
 'game_half',
 'quarter_end',
 'drive',
 'sp',
 'qtr',
 'down',
 'goal_to_go',
 'time',
 'yrdln',
 'ydstogo',
 'ydsnet',
 'desc',
 'play_type',
 'yards_gained',
 'shotgun',
 'no_huddle',
 'qb_dropback',
 'qb_kneel',
 'qb_spike',
 'qb_scramble',
 'pass_length',
 'pass_location',
 'air_yards',
 'yards_after_catch',
 'run_location',
 'run_gap',
 'field_goal_result',
 'kick_distance',
 'extra_point_result',
 'two_point_conv_result',
 'home_timeouts_remaining',
 'away_timeouts_remaining',
 'timeout',
 'timeout_team',
 'td_team',
 'td_player_name',
 'td_player_id',
 'posteam_timeouts_remaining',
 'defteam_timeouts_remaining',
 'total_home_score',
 'total_away_score',
 'posteam_score',
 'defteam_score',
 'score_differential',
 '

In [4]:
# Keep useful EPA columns
epa_cols = [
    "season",
    "week",
    "game_id",
    "posteam",
    "defteam",
    "play_type",
    "epa"
]

pbp_epa = pbp[epa_cols].copy()

pbp_epa.head()

,season,week,game_id,posteam,defteam,play_type,epa
0,2018,1,2018_01_ATL_PHI,None,None,None,-0.000000
1,2018,1,2018_01_ATL_PHI,ATL,PHI,kickoff,-0.000000
2,2018,1,2018_01_ATL_PHI,ATL,PHI,no_play,-0.773778
3,2018,1,2018_01_ATL_PHI,ATL,PHI,pass,0.850118
4,2018,1,2018_01_ATL_PHI,ATL,PHI,run,1.005722


In [5]:
# Filter to real offensive plays
pbp_epa = pbp_epa.dropna(subset=["posteam", "defteam", "epa"])

pbp_epa = pbp_epa[
    pbp_epa["play_type"].isin(["run", "pass"])
].copy()

pbp_epa.shape

(277516, 7)

In [6]:
# Create weekly offensive EPA
weekly_off_epa = (
    pbp_epa.groupby(["season", "week", "posteam"])
    .agg(
        off_epa_total=("epa", "sum"),
        off_plays=("epa", "count"),
        off_epa_per_play=("epa", "mean")
    )
    .reset_index()
)

weekly_off_epa = weekly_off_epa.rename(columns={"posteam": "team"})

weekly_off_epa.head()

,season,week,team,off_epa_total,off_plays,off_epa_per_play
0,2018,1,ARI,-12.141787,51,-0.238074
1,2018,1,ATL,-14.360798,65,-0.220935
2,2018,1,BAL,3.419919,75,0.045599
3,2018,1,BUF,-39.747009,61,-0.651590
4,2018,1,CAR,-2.003919,58,-0.034550


In [7]:
# Create weekly defensive EPA
weekly_def_epa = (
    pbp_epa.groupby(["season", "week", "defteam"])
    .agg(
        def_epa_allowed_total=("epa", "sum"),
        def_plays=("epa", "count"),
        def_epa_allowed_per_play=("epa", "mean")
    )
    .reset_index()
)

weekly_def_epa = weekly_def_epa.rename(columns={"defteam": "team"})

weekly_def_epa.head()

,season,week,team,def_epa_allowed_total,def_plays,def_epa_allowed_per_play
0,2018,1,ARI,12.512707,74,0.169091
1,2018,1,ATL,-5.225558,64,-0.081649
2,2018,1,BAL,-39.747009,61,-0.651590
3,2018,1,BUF,3.419919,75,0.045599
4,2018,1,CAR,-7.703186,57,-0.135144


In [8]:
# Merge offensive and defensive EPA
weekly_epa = weekly_off_epa.merge(
    weekly_def_epa,
    on=["season", "week", "team"],
    how="outer"
)

weekly_epa.head()

,season,week,team,off_epa_total,off_plays,off_epa_per_play,def_epa_allowed_total,def_plays,def_epa_allowed_per_play
0,2018,1,ARI,-12.141787,51,-0.238074,12.512707,74,0.169091
1,2018,1,ATL,-14.360798,65,-0.220935,-5.225558,64,-0.081649
2,2018,1,BAL,3.419919,75,0.045599,-39.747009,61,-0.651590
3,2018,1,BUF,-39.747009,61,-0.651590,3.419919,75,0.045599
4,2018,1,CAR,-2.003919,58,-0.034550,-7.703186,57,-0.135144


In [9]:
# Fill missing EPA values
weekly_epa = weekly_epa.fillna(0)

weekly_epa.head()

,season,week,team,off_epa_total,off_plays,off_epa_per_play,def_epa_allowed_total,def_plays,def_epa_allowed_per_play
0,2018,1,ARI,-12.141787,51,-0.238074,12.512707,74,0.169091
1,2018,1,ATL,-14.360798,65,-0.220935,-5.225558,64,-0.081649
2,2018,1,BAL,3.419919,75,0.045599,-39.747009,61,-0.651590
3,2018,1,BUF,-39.747009,61,-0.651590,3.419919,75,0.045599
4,2018,1,CAR,-2.003919,58,-0.034550,-7.703186,57,-0.135144


In [10]:
# Sort EPA data
weekly_epa = weekly_epa.sort_values(["team", "season", "week"]).reset_index(drop=True)

weekly_epa.head()

,season,week,team,off_epa_total,off_plays,off_epa_per_play,def_epa_allowed_total,def_plays,def_epa_allowed_per_play
0,2018,1,ARI,-12.141787,51,-0.238074,12.512707,74,0.169091
1,2018,2,ARI,-16.746893,43,-0.389463,13.658527,70,0.195122
2,2018,3,ARI,-10.980623,48,-0.228763,-3.970681,69,-0.057546
3,2018,4,ARI,-6.540291,56,-0.116791,1.771260,61,0.029037
4,2018,5,ARI,-6.187947,45,-0.137510,-16.357235,95,-0.172181


In [11]:
# Create EPA totals before each week
weekly_epa["games_with_epa_before"] = (
    weekly_epa.groupby(["team", "season"]).cumcount()
)

weekly_epa["off_epa_total_before"] = (
    weekly_epa.groupby(["team", "season"])["off_epa_total"]
    .transform(lambda x: x.cumsum().shift(1))
)

weekly_epa["off_plays_before"] = (
    weekly_epa.groupby(["team", "season"])["off_plays"]
    .transform(lambda x: x.cumsum().shift(1))
)

weekly_epa["def_epa_allowed_total_before"] = (
    weekly_epa.groupby(["team", "season"])["def_epa_allowed_total"]
    .transform(lambda x: x.cumsum().shift(1))
)

weekly_epa["def_plays_before"] = (
    weekly_epa.groupby(["team", "season"])["def_plays"]
    .transform(lambda x: x.cumsum().shift(1))
)

In [12]:
# Fill missing pre-game EPA totals
epa_before_cols = [
    "off_epa_total_before",
    "off_plays_before",
    "def_epa_allowed_total_before",
    "def_plays_before"
]

weekly_epa[epa_before_cols] = weekly_epa[epa_before_cols].fillna(0)

In [13]:
# Create pre-game EPA/play features
weekly_epa["off_epa_per_play_before"] = 0.0
weekly_epa["def_epa_allowed_per_play_before"] = 0.0

off_mask = weekly_epa["off_plays_before"] > 0
def_mask = weekly_epa["def_plays_before"] > 0

weekly_epa.loc[off_mask, "off_epa_per_play_before"] = (
    weekly_epa.loc[off_mask, "off_epa_total_before"] /
    weekly_epa.loc[off_mask, "off_plays_before"]
)

weekly_epa.loc[def_mask, "def_epa_allowed_per_play_before"] = (
    weekly_epa.loc[def_mask, "def_epa_allowed_total_before"] /
    weekly_epa.loc[def_mask, "def_plays_before"]
)

In [14]:
# Create last 3 week EPA features
weekly_epa["last3_off_epa_per_play"] = (
    weekly_epa.groupby(["team", "season"])["off_epa_per_play"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

weekly_epa["last3_def_epa_allowed_per_play"] = (
    weekly_epa.groupby(["team", "season"])["def_epa_allowed_per_play"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

weekly_epa[
    ["last3_off_epa_per_play", "last3_def_epa_allowed_per_play"]
] = weekly_epa[
    ["last3_off_epa_per_play", "last3_def_epa_allowed_per_play"]
].fillna(0)

In [15]:
# Keep EPA features only
epa_features = weekly_epa[
    [
        "season",
        "week",
        "team",
        "off_epa_per_play_before",
        "def_epa_allowed_per_play_before",
        "last3_off_epa_per_play",
        "last3_def_epa_allowed_per_play"
    ]
].copy()

epa_features.head()

,season,week,team,off_epa_per_play_before,def_epa_allowed_per_play_before,last3_off_epa_per_play,last3_def_epa_allowed_per_play
0,2018,1,ARI,0.000000,0.000000,0.000000,0.000000
1,2018,2,ARI,-0.238074,0.169091,-0.238074,0.169091
2,2018,3,ARI,-0.307326,0.181745,-0.313768,0.182106
3,2018,4,ARI,-0.280770,0.104228,-0.285433,0.102222
4,2018,5,ARI,-0.234392,0.087488,-0.245006,0.055538


In [16]:
# Save EPA features
os.makedirs("../data/processed", exist_ok=True)

epa_features.to_csv("../data/processed/epa_features_2018_2025.csv", index=False)

print("Saved EPA features.")
print(epa_features.shape)

Saved EPA features.
(4454, 7)


In [17]:
# Load expanded modeling dataset
modeling_data = pd.read_csv("../data/processed/modeling_dataset_expanded_2018_2025.csv")

modeling_data.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff,...,away_last3_avg_point_diff,away_last3_win_pct,avg_points_scored_diff,avg_points_allowed_diff,avg_point_diff_diff,win_pct_diff,last3_avg_points_scored_diff,last3_avg_points_allowed_diff,last3_avg_point_diff_diff,last3_win_pct_diff
0,2018,1,2018_01_ATL_PHI,2018-09-06,PHI,ATL,18.0,12.0,1,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2018,1,2018_01_BUF_BAL,2018-09-09,BAL,BUF,47.0,3.0,1,44.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2018,1,2018_01_PIT_CLE,2018-09-09,CLE,PIT,21.0,21.0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2018,1,2018_01_CIN_IND,2018-09-09,IND,CIN,23.0,34.0,0,-11.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2018,1,2018_01_TEN_MIA,2018-09-09,MIA,TEN,27.0,20.0,1,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
# Create home EPA features
home_epa = epa_features.rename(
    columns={
        "team": "home_team",
        "off_epa_per_play_before": "home_off_epa_per_play_before",
        "def_epa_allowed_per_play_before": "home_def_epa_allowed_per_play_before",
        "last3_off_epa_per_play": "home_last3_off_epa_per_play",
        "last3_def_epa_allowed_per_play": "home_last3_def_epa_allowed_per_play"
    }
)

In [19]:
# Create away EPA features
away_epa = epa_features.rename(
    columns={
        "team": "away_team",
        "off_epa_per_play_before": "away_off_epa_per_play_before",
        "def_epa_allowed_per_play_before": "away_def_epa_allowed_per_play_before",
        "last3_off_epa_per_play": "away_last3_off_epa_per_play",
        "last3_def_epa_allowed_per_play": "away_last3_def_epa_allowed_per_play"
    }
)

In [21]:
# Merge home EPA features
modeling_data_epa = modeling_data.merge(
    home_epa,
    on=["season", "week", "home_team"],
    how="left"
)

modeling_data_epa.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff,...,avg_point_diff_diff,win_pct_diff,last3_avg_points_scored_diff,last3_avg_points_allowed_diff,last3_avg_point_diff_diff,last3_win_pct_diff,home_off_epa_per_play_before,home_def_epa_allowed_per_play_before,home_last3_off_epa_per_play,home_last3_def_epa_allowed_per_play
0,2018,1,2018_01_ATL_PHI,2018-09-06,PHI,ATL,18.0,12.0,1,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2018,1,2018_01_BUF_BAL,2018-09-09,BAL,BUF,47.0,3.0,1,44.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2018,1,2018_01_PIT_CLE,2018-09-09,CLE,PIT,21.0,21.0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2018,1,2018_01_CIN_IND,2018-09-09,IND,CIN,23.0,34.0,0,-11.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2018,1,2018_01_TEN_MIA,2018-09-09,MIA,TEN,27.0,20.0,1,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
# Merge away EPA features
modeling_data_epa = modeling_data_epa.merge(
    away_epa,
    on=["season", "week", "away_team"],
    how="left"
)

modeling_data_epa.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff,...,last3_avg_point_diff_diff,last3_win_pct_diff,home_off_epa_per_play_before,home_def_epa_allowed_per_play_before,home_last3_off_epa_per_play,home_last3_def_epa_allowed_per_play,away_off_epa_per_play_before,away_def_epa_allowed_per_play_before,away_last3_off_epa_per_play,away_last3_def_epa_allowed_per_play
0,2018,1,2018_01_ATL_PHI,2018-09-06,PHI,ATL,18.0,12.0,1,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2018,1,2018_01_BUF_BAL,2018-09-09,BAL,BUF,47.0,3.0,1,44.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2018,1,2018_01_PIT_CLE,2018-09-09,CLE,PIT,21.0,21.0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2018,1,2018_01_CIN_IND,2018-09-09,IND,CIN,23.0,34.0,0,-11.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2018,1,2018_01_TEN_MIA,2018-09-09,MIA,TEN,27.0,20.0,1,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
# Check missing EPA values
epa_cols_check = [
    "home_off_epa_per_play_before",
    "home_def_epa_allowed_per_play_before",
    "home_last3_off_epa_per_play",
    "home_last3_def_epa_allowed_per_play",
    "away_off_epa_per_play_before",
    "away_def_epa_allowed_per_play_before",
    "away_last3_off_epa_per_play",
    "away_last3_def_epa_allowed_per_play"
]

modeling_data_epa[epa_cols_check].isna().sum()

home_off_epa_per_play_before            16
home_def_epa_allowed_per_play_before    16
home_last3_off_epa_per_play             16
home_last3_def_epa_allowed_per_play     16
away_off_epa_per_play_before            16
away_def_epa_allowed_per_play_before    16
away_last3_off_epa_per_play             16
away_last3_def_epa_allowed_per_play     16
dtype: int64

In [24]:
# Fill missing EPA values
modeling_data_epa[epa_cols_check] = modeling_data_epa[epa_cols_check].fillna(0)

In [25]:
# Create EPA difference features
modeling_data_epa["off_epa_per_play_diff"] = (
    modeling_data_epa["home_off_epa_per_play_before"] -
    modeling_data_epa["away_off_epa_per_play_before"]
)

modeling_data_epa["def_epa_allowed_per_play_diff"] = (
    modeling_data_epa["home_def_epa_allowed_per_play_before"] -
    modeling_data_epa["away_def_epa_allowed_per_play_before"]
)

modeling_data_epa["last3_off_epa_per_play_diff"] = (
    modeling_data_epa["home_last3_off_epa_per_play"] -
    modeling_data_epa["away_last3_off_epa_per_play"]
)

modeling_data_epa["last3_def_epa_allowed_per_play_diff"] = (
    modeling_data_epa["home_last3_def_epa_allowed_per_play"] -
    modeling_data_epa["away_last3_def_epa_allowed_per_play"]
)

In [26]:
# Save an EPA modeling dataset
modeling_data_epa.to_csv(
    "../data/processed/modeling_dataset_epa_2018_2025.csv",
    index=False
)

print("Saved EPA modeling dataset.")
print(modeling_data_epa.shape)

Saved EPA modeling dataset.
(2227, 48)


In [27]:
# Choose model features
features = [
    "avg_points_scored_diff",
    "avg_points_allowed_diff",
    "avg_point_diff_diff",
    "win_pct_diff",
    "last3_avg_points_scored_diff",
    "last3_avg_points_allowed_diff",
    "last3_avg_point_diff_diff",
    "last3_win_pct_diff",
    "off_epa_per_play_diff",
    "def_epa_allowed_per_play_diff",
    "last3_off_epa_per_play_diff",
    "last3_def_epa_allowed_per_play_diff"
]

target = "home_team_won"

In [28]:
# Train on 2018–2024 and test on 2025
train_data = modeling_data_epa[
    modeling_data_epa["season"].between(2018, 2024)
].copy()

test_data = modeling_data_epa[
    modeling_data_epa["season"] == 2025
].copy()

X_train = train_data[features]
y_train = train_data[target]

X_test = test_data[features]
y_test = test_data[target]

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 1942
Testing rows: 285


In [29]:
# Train model
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [30]:
# Predict
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [31]:
# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"EPA feature model accuracy: {accuracy:.2%}")
print("Previous best accuracy: 62.46%")

EPA feature model accuracy: 61.40%
Previous best accuracy: 62.46%


In [32]:
# Home-team baseline
home_team_baseline = [1] * len(y_test)

baseline_accuracy = accuracy_score(y_test, home_team_baseline)

print(f"Home-team baseline accuracy: {baseline_accuracy:.2%}")

Home-team baseline accuracy: 53.33%


In [33]:
# Classification report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.61      0.47      0.53       133
           1       0.61      0.74      0.67       152

    accuracy                           0.61       285
   macro avg       0.61      0.60      0.60       285
weighted avg       0.61      0.61      0.61       285



In [34]:
# Coefficents
coefficients = pd.DataFrame({
    "feature": features,
    "coefficient": model.coef_[0]
})

coefficients.sort_values("coefficient", ascending=False)

,feature,coefficient
8,off_epa_per_play_diff,1.356964
10,last3_off_epa_per_play_diff,0.514585
7,last3_win_pct_diff,0.449079
11,last3_def_epa_allowed_per_play_diff,0.072221
0,avg_points_scored_diff,0.016347
2,avg_point_diff_diff,0.015876
5,last3_avg_points_allowed_diff,0.002549
1,avg_points_allowed_diff,0.000470
4,last3_avg_points_scored_diff,-0.003085
6,last3_avg_point_diff_diff,-0.005633


In [35]:
# Create results table
results = test_data[
    [
        "season",
        "week",
        "game_id",
        "gameday",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "home_team_won"
    ]
].copy()

results["predicted_home_win"] = y_pred
results["home_win_probability"] = y_prob

results["predicted_winner"] = results.apply(
    lambda row: row["home_team"] if row["predicted_home_win"] == 1 else row["away_team"],
    axis=1
)

results["actual_winner"] = results.apply(
    lambda row: row["home_team"] if row["home_team_won"] == 1 else row["away_team"],
    axis=1
)

results["correct_prediction"] = (
    results["predicted_winner"] == results["actual_winner"]
)

results.head(10)

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,predicted_home_win,home_win_probability,predicted_winner,actual_winner,correct_prediction
1942,2025,1,2025_01_DAL_PHI,2025-09-04,PHI,DAL,24.0,20.0,1,1,0.54516,PHI,PHI,True
1943,2025,1,2025_01_KC_LAC,2025-09-05,LAC,KC,27.0,21.0,1,1,0.54516,LAC,LAC,True
1944,2025,1,2025_01_TB_ATL,2025-09-07,ATL,TB,20.0,23.0,0,1,0.54516,ATL,TB,False
1945,2025,1,2025_01_CIN_CLE,2025-09-07,CLE,CIN,16.0,17.0,0,1,0.54516,CLE,CIN,False
1946,2025,1,2025_01_MIA_IND,2025-09-07,IND,MIA,33.0,8.0,1,1,0.54516,IND,IND,True
1947,2025,1,2025_01_CAR_JAX,2025-09-07,JAX,CAR,26.0,10.0,1,1,0.54516,JAX,JAX,True
1948,2025,1,2025_01_LV_NE,2025-09-07,NE,LV,13.0,20.0,0,1,0.54516,NE,LV,False
1949,2025,1,2025_01_ARI_NO,2025-09-07,NO,ARI,13.0,20.0,0,1,0.54516,NO,ARI,False
1950,2025,1,2025_01_PIT_NYJ,2025-09-07,NYJ,PIT,32.0,34.0,0,1,0.54516,NYJ,PIT,False
1951,2025,1,2025_01_NYG_WAS,2025-09-07,WAS,NYG,21.0,6.0,1,1,0.54516,WAS,WAS,True


In [36]:
# Accuracy be week
accuracy_by_week = (
    results.groupby("week")["correct_prediction"]
    .mean()
    .reset_index()
)

accuracy_by_week["accuracy_percent"] = (
    accuracy_by_week["correct_prediction"] * 100
)

accuracy_by_week = accuracy_by_week.drop(columns=["correct_prediction"])

accuracy_by_week

,week,accuracy_percent
0,1,56.250000
1,2,75.000000
2,3,50.000000
3,4,56.250000
4,5,50.000000
5,6,53.333333
6,7,73.333333
7,8,69.230769
8,9,50.000000
9,10,78.571429


In [37]:
# Accuracy by confidence
def get_confidence(prob):
    if prob >= 0.65 or prob <= 0.35:
        return "High"
    elif prob >= 0.57 or prob <= 0.43:
        return "Medium"
    else:
        return "Low"

results["confidence"] = results["home_win_probability"].apply(get_confidence)

confidence_accuracy = (
    results.groupby("confidence")["correct_prediction"]
    .agg(["count", "mean"])
    .reset_index()
)

confidence_accuracy["accuracy_percent"] = confidence_accuracy["mean"] * 100

confidence_accuracy = confidence_accuracy.rename(
    columns={"count": "number_of_games"}
)

confidence_accuracy = confidence_accuracy.drop(columns=["mean"])

confidence_accuracy

,confidence,number_of_games,accuracy_percent
0,High,87,71.264368
1,Low,115,56.521739
2,Medium,83,57.831325


In [38]:
# Save EPA predictions
os.makedirs("../data/predictions", exist_ok=True)

results.to_csv(
    "../data/predictions/epa_model_test_predictions.csv",
    index=False
)

print("Saved EPA model test predictions.")
print(results.shape)

Saved EPA model test predictions.
(285, 15)


In [39]:
# Final summary numbers
print("EPA feature model accuracy:", f"{accuracy:.2%}")
print("Previous best accuracy: 62.46%")
print("Home-team baseline accuracy:", f"{baseline_accuracy:.2%}")
print("Number of test games:", len(results))
print("Correct predictions:", results["correct_prediction"].sum())
print("Incorrect predictions:", len(results) - results["correct_prediction"].sum())

EPA feature model accuracy: 61.40%
Previous best accuracy: 62.46%
Home-team baseline accuracy: 53.33%
Number of test games: 285
Correct predictions: 175
Incorrect predictions: 110


## Summary

In this notebook, I added EPA/play features to the NFL forecasting model.

EPA features are useful because they measure play-level efficiency instead of only using final scores or win percentage. The model now includes offensive EPA/play, defensive EPA allowed/play, and last-3-game EPA features.

The model trained on 2018–2024 and tested on 2025, using the same realistic season-based evaluation setup as the previous notebooks.

The next step is to compare this EPA model accuracy to the previous best accuracy of 62.46%.